# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

In [5]:
# ====================
# Important Libraries
# ====================
%pip install numpy
%pip install pandas
%pip install matplotlib
%pip install -q control

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import control

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [36]:
# ==========================================
# Part 1: Reconstruct the Control Matrix K
# ==========================================
# Use: model-A.tsv, model-B.tsv, cost-Q.tsv, cost-R.tsv
A_df = pd.read_csv("model-A.tsv", sep="\t")
A = A_df[['x', 'v', 'a']].to_numpy()

B_df = pd.read_csv("model-B.tsv", sep="\t")
B = B_df[['u']].to_numpy()

Q_df = pd.read_csv("cost-Q.tsv", sep="\t")
Q = Q_df[['x', 'v', 'a']].to_numpy()

R_df = pd.read_csv("cost-R.tsv", sep="\t")
R = R_df[['u']].to_numpy()


# K for optimal action selection
K, _, _ = control.dlqr(A, B, Q, R)
K

array([[0.33445985, 1.30445413, 1.85813088]])

Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [ ]:
# ==================================================
# Save K to TSV file named 'control-K-intended.tsv'
# ==================================================
control_K_intended = pd.DataFrame({
    'x': K[:, 0],
    'v': K[:, 1],
    'a': K[:, 2]
    })  

# Save file
control_K_intended.to_csv('control-K-intended.tsv', sep='\t', index=False)
control_K_intended

,x,v,a
0,0.33446,1.304454,1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [40]:
# =====================================================
# Part 2: Recompute the actions for the logged states
# =====================================================
# Use: qc-log.tsv, u_t = -K_t x_t.
qc_log = pd.read_csv("qc-log.tsv", sep="\t")

logged_states = qc_log[['x', 'v', 'a']].to_numpy()
# print(logged_states)

u = -(logged_states @ K.T).ravel()


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [ ]:
# =======================================================
# Save computed actions to TSV file named 'qc-check.tsv'
# =======================================================
qc_check = pd.DataFrame({
    "index": qc_log["index"],
    "u_check": u,
    })

# Save file
qc_check.to_csv("qc-check.tsv", sep="\t", index=False)
qc_check

,index,u_check
0,0,1.672299e+00
1,1,-1.152095e+00
2,2,-1.235183e+00
3,3,-2.885035e-01
4,4,3.692016e-01
...,...,...
95,95,-5.148677e-19
96,96,2.552224e-19
97,97,3.971337e-19
98,98,1.886533e-19


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [45]:
# ====================================================
# Part 3: Reverse Engineer the Actual Control Matrix
# ====================================================
qc_log = pd.read_csv("qc-log.tsv", sep="\t")
qc_check = pd.read_csv("qc-check.tsv", sep="\t")

logged_states = qc_log[['x', 'v', 'a']].to_numpy()
u = qc_check['u_check'].to_numpy()

# Solve for K using least squares
K_actual, _, _, _ = np.linalg.lstsq(logged_states, -u, rcond=None)
K_actual = K_actual.T
K_actual

array([0.33445985, 1.30445413, 1.85813088])

Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [ ]:
# =========================================================
# Save K-actual to TSV file named 'control-K-actual.tsv'
# =========================================================
control_K_actual = pd.DataFrame({
    'x': [K_actual[0]],
    'v': [K_actual[1]],
    'a': [K_actual[2]]
})
    
# Save file
control_K_actual.to_csv("control-K-actual.tsv", sep="\t", index=False)
control_K_actual

,x,v,a
0,0.33446,1.304454,1.858131


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [ ]:
# YOUR CHANGES HERE

...

Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [ ]:
# YOUR CHANGES HERE

...

Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.